# Discovery - Limpieza de Datos

## Inspeción preliminar

In [4]:
import pandas as pd
import pathlib
import re
import unicodedata
import ast
import numpy as np

In [5]:
# Ruta recomendada para compartir el proyecto sin versionar datos sensibles.
path_todos_registros = pathlib.Path("../data_raw/todos_registros.xlsx")

# Fallback para el entorno local original del proyecto.
if not path_todos_registros.exists():
    path_todos_registros = pathlib.Path(r"D:\Open_Vzla_SOS_Calidad_de_Datos\aevscraping\datos_consolidados\todos_registros.xlsx")

df_raw = pd.read_excel(path_todos_registros)

# Conservamos el scraping original intacto y trabajamos siempre sobre una copia.
df_clean = df_raw.copy()
df_clean.head()


,ID,Nombre,Cédula,Edad,Última Ubicación,Teléfono Contacto,Observaciones,Estado,Ubicación Encontrado,Encontrado Por,Cédula Encontrado,URL Foto,Fecha Registro,Fecha Actualización,Es Menor,Fuente
0,7410076847799689,José Gregorio Rojas Bello,No registrado,NaN,La Guaira,+584142504728,No sabemos d el y su familia padre esposa hijos,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/7ec6...,2026-06-28T17:50:42.114Z,2026-06-28T17:50:42.114Z,No,venezuelatebusca
1,2590484473230077,Hector Reyes,No registrado,NaN,San Agustin,+584124989844,Acabo de ver un vídeo de WhatsApp donde tienen...,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/cc73...,2026-06-28T17:50:34.923Z,2026-06-28T17:50:34.923Z,No,venezuelatebusca
2,152612323455194,Neiderlyn Alfonso,No registrado,6.0,Tanaguarenas la Guaira los cocos,+584241592570,NaN,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/ba9b...,2026-06-28T17:49:37.868Z,2026-06-28T17:49:37.868Z,Sí,venezuelatebusca
3,2394053969385436,Samuel Vicente Cortez Olivier,37099131,11.0,"Conjunto Residencial Caribe, la Guaira",+584127387227,Pantalon azul franela,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/53e6...,2026-06-28T17:48:07.144Z,2026-06-28T17:48:07.144Z,Sí,venezuelatebusca
4,6937486286932653,Amir Infante Galván,No registrado,16.0,La guaira,+584162443424,NaN,Desaparecido,NaN,NaN,NaN,NaN,2026-06-28T17:48:04.530Z,2026-06-28T17:48:04.530Z,Sí,venezuelatebusca


In [6]:
# Filas y columnas del DataFrame
df_clean.shape

(241463, 16)

In [7]:
# Información general del DataFrame
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 241463 entries, 0 to 241462
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   ID                    241463 non-null  int64  
 1   Nombre                241463 non-null  str    
 2   Cédula                241463 non-null  str    
 3   Edad                  140396 non-null  float64
 4   Última Ubicación      228992 non-null  str    
 5   Teléfono Contacto     101095 non-null  str    
 6   Observaciones         162535 non-null  str    
 7   Estado                241463 non-null  str    
 8   Ubicación Encontrado  14678 non-null   str    
 9   Encontrado Por        16091 non-null   str    
 10  Cédula Encontrado     0 non-null       float64
 11  URL Foto              129962 non-null  str    
 12  Fecha Registro        241463 non-null  str    
 13  Fecha Actualización   241463 non-null  str    
 14  Es Menor              241463 non-null  str    
 15  Fuente     

## Normalización

In [8]:
# Función para quitar acentos
def quitar_acentos(texto):
    if pd.isna(texto):
        return pd.NA

    return ''.join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )

In [9]:
# Función para normalizar texto
# Centralizamos la limpieza básica para aplicar el mismo criterio a todas las columnas textuales.
def normalizar_texto(texto):
    if pd.isna(texto):
        return pd.NA

    texto = str(texto)
    texto = quitar_acentos(texto)
    texto = texto.strip()
    texto = re.sub(r"\s+", " ", texto)

    return texto

In [10]:
# Normalizar columnas de texto
columnas_texto = [
    "Nombre",
    "Última Ubicación",
    "Teléfono Contacto",
    "Observaciones",
    "Estado",
    "Ubicación Encontrado",
    "Encontrado Por",
    "URL Foto",
    "Fuente"
]

for col in columnas_texto:
    df_clean[col] = df_clean[col].apply(normalizar_texto)

In [11]:
# Normalizar la cédula
df_clean["Cédula"] = (
    df_clean["Cédula"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
)

In [12]:
df_clean["Teléfono Contacto"] = (
    df_clean["Teléfono Contacto"]
    .str.replace(r"\D", "", regex=True)
)

In [13]:
# Convertir fechas
df_clean["Fecha Registro"] = pd.to_datetime(
    df_clean["Fecha Registro"],
    errors="coerce"
)

df_clean["Fecha Actualización"] = pd.to_datetime(
    df_clean["Fecha Actualización"],
    errors="coerce"
)

In [14]:
# Edad
df_clean["Edad"] = pd.to_numeric(
    df_clean["Edad"],
    errors="coerce"
)

In [15]:
# Es Menor
df_clean["Es Menor"] = (
    df_clean["Es Menor"]
    .str.strip()
    .str.title()
)


In [16]:
# Estado
df_clean["Estado"] = (
    df_clean["Estado"]
    .str.strip()
    .str.title()
)

In [17]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 241463 entries, 0 to 241462
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   ID                    241463 non-null  int64              
 1   Nombre                241463 non-null  str                
 2   Cédula                241463 non-null  str                
 3   Edad                  140396 non-null  float64            
 4   Última Ubicación      228992 non-null  str                
 5   Teléfono Contacto     101095 non-null  str                
 6   Observaciones         162535 non-null  str                
 7   Estado                241463 non-null  str                
 8   Ubicación Encontrado  14678 non-null   str                
 9   Encontrado Por        16091 non-null   str                
 10  Cédula Encontrado     0 non-null       float64            
 11  URL Foto              129962 non-null  str                
 12 

## Validación de los datos

In [18]:
df_validated = df_clean.copy()

In [19]:
# Validar cédulas
mask = (
    df_validated["Cédula"]
    .str.fullmatch(r"\d{7,8}", na=False)
)

df_validated.loc[~mask, "Cédula"] = pd.NA

In [20]:
df_validated["Cédula"].str.len().value_counts(dropna=False)

Cédula
NaN    230767
8.0      8065
7.0      2631
Name: count, dtype: int64

In [21]:
# Validar teléfonos
mask = (
    df_validated["Teléfono Contacto"]
    .str.fullmatch(r"\d{10,15}", na=False)
)

df_validated.loc[~mask, "Teléfono Contacto"] = pd.NA

In [22]:
# Validar edad
mask = df_validated["Edad"].between(0, 110)

df_validated.loc[~mask, "Edad"] = pd.NA

In [23]:
# Validar URLs
mask = (
    df_validated["URL Foto"]
    .str.match(r"^https?://", na=False)
)

df_validated.loc[~mask, "URL Foto"] = pd.NA

In [24]:
# Validar Estado
df_validated["Estado"].value_counts(dropna=False)

Estado
Desaparecido    203737
Localizado       37726
Name: count, dtype: int64

In [25]:
estados_validos = {
    "Localizado",
    "Desaparecido"
}

mask = df_validated["Estado"].isin(estados_validos)

df_validated.loc[~mask, "Estado"] = pd.NA

In [26]:
# Cambiar el tipo str a string para manejo de los valores faltantes para evitar inconsistencias con NaN, None y cadenas vacías.
columnas_texto = [
    "Nombre",
    "Cédula",
    "Última Ubicación",
    "Teléfono Contacto",
    "Observaciones",
    "Estado",
    "Ubicación Encontrado",
    "Encontrado Por",
    "URL Foto",
    "Es Menor",
    "Fuente"
]

df_validated[columnas_texto] = (
    df_validated[columnas_texto]
    .astype("string")
)

In [27]:
df_validated.info()

<class 'pandas.DataFrame'>
RangeIndex: 241463 entries, 0 to 241462
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   ID                    241463 non-null  int64              
 1   Nombre                241463 non-null  string             
 2   Cédula                10696 non-null   string             
 3   Edad                  140391 non-null  float64            
 4   Última Ubicación      228992 non-null  string             
 5   Teléfono Contacto     86550 non-null   string             
 6   Observaciones         162535 non-null  string             
 7   Estado                241463 non-null  string             
 8   Ubicación Encontrado  14678 non-null   string             
 9   Encontrado Por        16091 non-null   string             
 10  Cédula Encontrado     0 non-null       float64            
 11  URL Foto              129962 non-null  string             
 12 

## Entities

In [28]:
# Copia base
df_entities = df_validated.copy()

In [29]:
#  Función de limpieza 
def limpiar_nombre(texto):
    if pd.isna(texto):
        return pd.NA

    texto = str(texto)

    # eliminar HTML
    texto = re.sub(r"<.*?>", "", texto)

    # eliminar XSS / scripts
    texto = re.sub(r"(?i)(on\w+=|javascript:|script)", "", texto)

    # normalización unicode
    texto = unicodedata.normalize("NFKD", texto)

    # solo letras y espacios
    texto = re.sub(r"[^a-zA-ZÀ-ÿñÑ\s]", "", texto)

    # espacios limpios
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto if texto != "" else pd.NA

In [30]:
# Crear columna limpia
df_entities["Nombre_clean"] = df_entities["Nombre"].apply(limpiar_nombre)

In [31]:
# Inspección de datos válidos
df_validos = df_entities[df_entities["Nombre_clean"].notna()].copy()
df_validos.info()

<class 'pandas.DataFrame'>
Index: 238235 entries, 0 to 241462
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   ID                    238235 non-null  int64              
 1   Nombre                238235 non-null  string             
 2   Cédula                10696 non-null   string             
 3   Edad                  140205 non-null  float64            
 4   Última Ubicación      225765 non-null  string             
 5   Teléfono Contacto     86543 non-null   string             
 6   Observaciones         159775 non-null  string             
 7   Estado                238235 non-null  string             
 8   Ubicación Encontrado  14678 non-null   string             
 9   Encontrado Por        16091 non-null   string             
 10  Cédula Encontrado     0 non-null       float64            
 11  URL Foto              127027 non-null  string             
 12  Fech

In [32]:
# Filtrar nombres válidos 
df_validos[df_validos["Nombre_clean"].notna()].head()

,ID,Nombre,Cédula,Edad,Última Ubicación,Teléfono Contacto,Observaciones,Estado,Ubicación Encontrado,Encontrado Por,Cédula Encontrado,URL Foto,Fecha Registro,Fecha Actualización,Es Menor,Fuente,Nombre_clean
0,7410076847799689,Jose Gregorio Rojas Bello,<NA>,NaN,La Guaira,584142504728,No sabemos d el y su familia padre esposa hijos,Desaparecido,<NA>,<NA>,NaN,https://venezuelatebusca.com/media/photos/7ec6...,2026-06-28 17:50:42.114000+00:00,2026-06-28 17:50:42.114000+00:00,No,venezuelatebusca,Jose Gregorio Rojas Bello
1,2590484473230077,Hector Reyes,<NA>,NaN,San Agustin,584124989844,Acabo de ver un video de WhatsApp donde tienen...,Desaparecido,<NA>,<NA>,NaN,https://venezuelatebusca.com/media/photos/cc73...,2026-06-28 17:50:34.923000+00:00,2026-06-28 17:50:34.923000+00:00,No,venezuelatebusca,Hector Reyes
2,152612323455194,Neiderlyn Alfonso,<NA>,6.0,Tanaguarenas la Guaira los cocos,584241592570,<NA>,Desaparecido,<NA>,<NA>,NaN,https://venezuelatebusca.com/media/photos/ba9b...,2026-06-28 17:49:37.868000+00:00,2026-06-28 17:49:37.868000+00:00,Sí,venezuelatebusca,Neiderlyn Alfonso
3,2394053969385436,Samuel Vicente Cortez Olivier,37099131,11.0,"Conjunto Residencial Caribe, la Guaira",584127387227,Pantalon azul franela,Desaparecido,<NA>,<NA>,NaN,https://venezuelatebusca.com/media/photos/53e6...,2026-06-28 17:48:07.144000+00:00,2026-06-28 17:48:07.144000+00:00,Sí,venezuelatebusca,Samuel Vicente Cortez Olivier
4,6937486286932653,Amir Infante Galvan,<NA>,16.0,La guaira,584162443424,<NA>,Desaparecido,<NA>,<NA>,NaN,<NA>,2026-06-28 17:48:04.530000+00:00,2026-06-28 17:48:04.530000+00:00,Sí,venezuelatebusca,Amir Infante Galvan


In [33]:
df_validos["Nombre_clean"].value_counts().head(20)

Nombre_clean
ll                    29625
J                     19348
a                     14942
NN                    10127
d                      2639
L                      1857
Yo                     1729
F                       221
Clementina Segovia       24
Zulay Sarmiento          23
Vicky Urdaneta           22
Osmary Montilla          21
Carlos Gonzalez          21
Alicia Falcon            21
Gustavo Ruiz             21
Nancy Hernandez          21
Luis Auyanet             20
Camila Gutierrez         20
Eliana Mogollon          19
Orlando Gonzalez         19
Name: count, dtype: int64

In [34]:
# Filtro básico de calidad de nombre
df_validos = df_validos[
    df_validos["Nombre_clean"].notna() &
    (df_validos["Nombre_clean"].str.len() > 3)
]

In [35]:
# Eliminar patrones basura
basura = {"nn", "na", "ll", "j", "a", "d", "l", "yo"}

df_validos = df_validos[
    ~df_validos["Nombre_clean"].str.lower().isin(basura)
]

In [36]:
# Detectar “parece nombre real”
df_validos = df_validos[
    df_validos["Nombre_clean"].str.contains(r"^[A-Za-zÁÉÍÓÚÑáéíóúñ ]{4,}$", regex=True)
]

In [37]:
df_validos["Nombre_clean"]

0                Jose Gregorio Rojas Bello
1                             Hector Reyes
2                        Neiderlyn Alfonso
3            Samuel Vicente Cortez Olivier
4                      Amir Infante Galvan
                        ...               
241458                          Maria mata
241459                   rodrigo rodriguez
241460    Skarlent Rodriguez y Jose Castro
241461                        Jesus sabala
241462                    Liliagnit Borges
Name: Nombre_clean, Length: 157700, dtype: str

## Duplicados

In [38]:
# Duplicados exactos (fila completa)
df_validos.duplicated().sum()



np.int64(0)

In [39]:
# Filtrar solo cédulas válidas (no NaN) y duplicadas
df_duplicados = df_validos[
    df_validos["Cédula"].notna() &
    df_validos["Cédula"].duplicated(keep=False)
].sort_values("Cédula")

df_duplicados

,ID,Nombre,Cédula,Edad,Última Ubicación,Teléfono Contacto,Observaciones,Estado,Ubicación Encontrado,Encontrado Por,Cédula Encontrado,URL Foto,Fecha Registro,Fecha Actualización,Es Menor,Fuente,Nombre_clean
2738,5214567643934092,Maria Laura Hernandez Acosta,17307478,42.0,Playa grande la guaira | Hotel Catimar,34697293314,"No se sabe | Mujer de 1,70 con mechas. Utiliza...",Desaparecido,<NA>,<NA>,NaN,https://venezuelatebusca.com/media/photos/5944...,2026-06-28 02:19:45.262000+00:00,2026-06-28 12:11:22.592000+00:00,No,venezuelatebusca,Maria Laura Hernandez Acosta
17161,6308673208560697,Maria Laura Hernandez Acosta,17307478,42.0,Hotel Catimar,584145198367,"Mujer de 1,70 con mechas. Utiliza reloj negro ...",Desaparecido,<NA>,<NA>,NaN,<NA>,2026-06-25 22:17:37.075000+00:00,2026-06-26 11:25:48.999000+00:00,No,<NA>,Maria Laura Hernandez Acosta
3168,3271040832349847,Leslie Zaragoza,37425922,10.0,La guaira los corales vivienda vene8,584165220398,Catirita ojos ambar delgadita cabello color ch...,Desaparecido,<NA>,<NA>,NaN,https://venezuelatebusca.com/media/photos/166a...,2026-06-27 19:57:41.877000+00:00,2026-06-28 02:00:30.978000+00:00,Sí,venezuelatebusca,Leslie Zaragoza
8426,8705332788312286,Leslie Zaragoza,37425922,10.0,La Guaira - caribe,04128983063,<NA>,Desaparecido,<NA>,<NA>,NaN,https://venezuelatebusca.com/media/photos/52fa...,2026-06-26 17:50:00.779000+00:00,2026-06-26 17:50:00.779000+00:00,Sí,<NA>,Leslie Zaragoza
2418,7758004926016465,Carmen Josefa Guzman Palacios,5491667,68.0,Edificio tahiti o Tamiami Caraballeda | La gua...,584241373304,"Mujer de 1.60 metros aproximadamente, gordita,...",Desaparecido,<NA>,<NA>,NaN,https://venezuelatebusca.com/media/photos/da8a...,2026-06-28 05:24:59.194000+00:00,2026-06-28 12:11:22.592000+00:00,No,venezuelatebusca,Carmen Josefa Guzman Palacios
31856,5221351944823551,Carmen Josefa Guzman Palacios,5491667,68.0,La guaira caraballeda,525534762784,"Mujer de 1.60 metros aproximadamente, gordita,...",Desaparecido,<NA>,<NA>,NaN,<NA>,2026-06-25 08:05:20.587000+00:00,2026-06-26 11:25:48.999000+00:00,No,<NA>,Carmen Josefa Guzman Palacios


In [40]:
# Función para decidir qué registro conservar de cada cédula
def seleccionar_registro(grupo):
    # Regla 1: si existe algún "Localizado", conservar el primero
    localizados = grupo[grupo["Estado"] == "Localizado"]
    if not localizados.empty:
        return localizados.index[0]

    # Regla 2: si todos son "Desaparecido", priorizar el que tenga teléfono
    if (grupo["Estado"] == "Desaparecido").all():
        con_telefono = grupo[grupo["Teléfono Contacto"].notna()]
        if not con_telefono.empty:
            return con_telefono.index[0]

    # Si no se cumple ninguna regla, conservar el primer registro
    return grupo.index[0]


# Índices de las filas que se conservarán
indices_conservar = (
    df_validos[df_validos["Cédula"].notna()]
    .groupby("Cédula", group_keys=False)
    .apply(seleccionar_registro)
)

# Cédulas que aparecen una sola vez
indices_unicos = df_validos[
    df_validos["Cédula"].isna() |
    ~df_validos["Cédula"].duplicated(keep=False)
].index

# DataFrame final
df_validos = df_validos.loc[
    indices_unicos.union(indices_conservar)
].sort_index().reset_index(drop=True)

In [41]:
df_validos.info()

<class 'pandas.DataFrame'>
RangeIndex: 157697 entries, 0 to 157696
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   ID                    157697 non-null  int64              
 1   Nombre                157697 non-null  string             
 2   Cédula                10693 non-null   string             
 3   Edad                  121640 non-null  float64            
 4   Última Ubicación      145227 non-null  string             
 5   Teléfono Contacto     86519 non-null   string             
 6   Observaciones         98599 non-null   string             
 7   Estado                157697 non-null  string             
 8   Ubicación Encontrado  14674 non-null   string             
 9   Encontrado Por        16087 non-null   string             
 10  Cédula Encontrado     0 non-null       float64            
 11  URL Foto              113056 non-null  string             
 12 

In [42]:
df_validos["Nombre_clean"].value_counts().head(20)

Nombre_clean
Clementina Segovia    24
Zulay Sarmiento       23
Vicky Urdaneta        22
Osmary Montilla       21
Carlos Gonzalez       21
Alicia Falcon         21
Gustavo Ruiz          21
Nancy Hernandez       21
Luis Auyanet          20
Camila Gutierrez      20
Eliana Mogollon       19
Orlando Gonzalez      19
Gabriel Gonzalez      19
Melvis Gonzalez       19
Andres Poleo          19
Ysmael Pena Perez     18
Guillermo Marquina    18
Pablo Armas           18
Elisa Sciamanna       18
Carlos Torres         18
Name: count, dtype: int64

In [43]:
# Crear un identificador único por fila
df_entidad = df_validos.copy()


In [44]:
df_entidad["fila_id"] = np.arange(len(df_entidad))

In [45]:
# Inicializar Union-Find (Disjoint Set)
padre = np.arange(len(df_entities))

def buscar(x):
    while padre[x] != x:
        padre[x] = padre[padre[x]]
        x = padre[x]
    return x

def unir(a, b):
    ra = buscar(a)
    rb = buscar(b)
    if ra != rb:
        padre[rb] = ra

In [46]:
# Unir por Cédula
for _, grupo in (
    df_entidad[df_entidad["Cédula"].notna()]
    .groupby("Cédula")
):
    filas = grupo["fila_id"].to_numpy()
    for i in range(1, len(filas)):
        unir(filas[0], filas[i])

In [47]:
# Unir por Teléfono
for _, grupo in (
    df_entidad[df_entidad["Teléfono Contacto"].notna()]
    .groupby("Teléfono Contacto")
):
    filas = grupo["fila_id"].to_numpy()
    for i in range(1, len(filas)):
        unir(filas[0], filas[i])

In [48]:
# Unir por URL Foto
for _, grupo in (
    df_entidad[df_entidad["URL Foto"].notna()]
    .groupby("URL Foto")
):
    filas = grupo["fila_id"].to_numpy()
    for i in range(1, len(filas)):
        unir(filas[0], filas[i])

In [49]:
from difflib import SequenceMatcher

def similitud(a, b):
    if pd.isna(a) or pd.isna(b):
        return 0
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()


for _, grupo in df_entidad.groupby("Nombre_clean"):

    if len(grupo) < 2:
        continue

    filas = grupo.to_dict("records")

    for i in range(len(filas)):
        for j in range(i + 1, len(filas)):

            score = 0

            # Cédula
            if (
                pd.notna(filas[i]["Cédula"]) and
                filas[i]["Cédula"] == filas[j]["Cédula"]
            ):
                score += 100

            # Teléfono
            if (
                pd.notna(filas[i]["Teléfono Contacto"]) and
                filas[i]["Teléfono Contacto"] == filas[j]["Teléfono Contacto"]
            ):
                score += 100

            # Edad
            e1 = filas[i]["Edad"]
            e2 = filas[j]["Edad"]

            if pd.notna(e1) and pd.notna(e2):
                if abs(e1 - e2) <= 2:
                    score += 20
            else:
                score += 10

            # Ubicación
            if similitud(
                filas[i]["Última Ubicación"],
                filas[j]["Última Ubicación"]
            ) >= 0.60:
                score += 30

            # Estado
            if filas[i]["Estado"] == filas[j]["Estado"]:
                score += 5

            if score >= 40:
                unir(filas[i]["fila_id"], filas[j]["fila_id"])

In [50]:
# Crear el entity_id
df_entidad["entity_id"] = [
    buscar(i) for i in df_entidad["fila_id"]
]

df_entidad["entity_id"] = (
    pd.factorize(df_entidad["entity_id"])[0]
)

In [51]:
df_entidad["score"] = (
    (df_entidad["Estado"].eq("Localizado").astype(int) * 1000) +
    (df_entidad["Cédula"].notna().astype(int) * 100) +
    (df_entidad["Teléfono Contacto"].notna().astype(int) * 50) +
    (df_entidad["URL Foto"].notna().astype(int) * 20) +
    (df_entidad["Edad"].notna().astype(int) * 10)
)

In [52]:
df_entidad = df_entidad.sort_values(
    ["entity_id", "score", "Fecha Actualización"],
    ascending=[True, False, False]
)

In [53]:
df_final = (
    df_entidad
    .drop_duplicates(subset="entity_id", keep="first")
    .reset_index(drop=True)
)

In [54]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 79451 entries, 0 to 79450
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   ID                    79451 non-null  int64              
 1   Nombre                79451 non-null  string             
 2   Cédula                8685 non-null   string             
 3   Edad                  61681 non-null  float64            
 4   Última Ubicación      71319 non-null  string             
 5   Teléfono Contacto     51912 non-null  string             
 6   Observaciones         49781 non-null  string             
 7   Estado                79451 non-null  string             
 8   Ubicación Encontrado  10489 non-null  string             
 9   Encontrado Por        11754 non-null  string             
 10  Cédula Encontrado     0 non-null      float64            
 11  URL Foto              54799 non-null  string             
 12  Fecha Registro 

In [55]:
df_final["Nombre_clean"].value_counts().head(20)

Nombre_clean
Carlos Hernandez     12
Daniel Martinez      11
Carlos Gonzalez      11
Luis Rodriguez       10
Jean Tuarez          10
Carlos Lopez         10
Jose Garcia          10
Javier Hernandez     10
Maria Castillo       10
Jesus Rodriguez       9
Jose Rodriguez        9
Jose Escobar          9
Laura Granatti        9
Hector Escalona       9
Ramon Perez           9
Blanca Bustamante     9
Jose Perez            8
Ana Salazar           8
Genesis Rodriguez     8
Luis Perez            8
Name: count, dtype: int64

In [56]:
def separar_personas(nombre):
    """
    Recibe un nombre y devuelve una lista de personas.
    Si no encuentra varios nombres devuelve una lista con un solo elemento.
    """

    if pd.isna(nombre):
        return []

    nombre = str(nombre).strip()

    # separadores considerados
    patron = r"\s+(?:y|e|&|/)\s+|;"

    personas = re.split(patron, nombre)

    # eliminar espacios y elementos vacíos
    personas = [
        p.strip()
        for p in personas
        if p.strip()
    ]

    return personas

In [57]:
nuevas_filas = []

for _, fila in df_final.iterrows():

    personas = separar_personas(fila["Nombre_clean"])

    if len(personas) == 1:
        nuevas_filas.append(fila.copy())

    else:
        for persona in personas:
            nueva = fila.copy()
            nueva["Nombre_clean"] = persona
            nueva["Nombre"] = persona
            nuevas_filas.append(nueva)

df_final = pd.DataFrame(nuevas_filas).reset_index(drop=True)